In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "accelerate", "torchaudio", "soundfile", "librosa"])

import os, json, glob, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: enable GPU (Settings -> Accelerator -> GPU T4 x2) and restart.")

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

Device: cuda:0


In [2]:
CM_ROOT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"
FT_ROOT = "/kaggle/input/datasets/ajfaisal002/fakett-dataset"

print("=== Crossmodal dataset ===")
for root, dirs, files in os.walk(CM_ROOT):
    level = root.replace(CM_ROOT, "").count(os.sep)
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")
        for f in files[:3]:
            print("  " * (level+1) + f)

print("\n=== FakeTT dataset ===")
for root, dirs, files in os.walk(FT_ROOT):
    level = root.replace(FT_ROOT, "").count(os.sep)
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")
        for f in files[:3]:
            print("  " * (level+1) + f)

=== Crossmodal dataset ===
crossmodal-misleading-video-dataset/
  extracted_audio/
    extracted_audio/
  dataset_kaggle/
    annotations/
      test_annotations.json
      train_annotations.json
    extracted_frames/
    extracted_text/
      train_vision_text.jsonl
      test_vision_text.jsonl
      train_metadata.csv

=== FakeTT dataset ===
fakett-dataset/
  data.json
  video/
    7276878408070483246.mp4
    7339164334095961387.mp4
    7294301911006711086.mp4


# Cell 3 — Load frozen encoders (CLIP, Wav2Vec2, XLM-R, Qwen)

In [3]:
from transformers import (
    CLIPModel, CLIPProcessor,
    Wav2Vec2Model, Wav2Vec2FeatureExtractor,
    AutoModel, AutoTokenizer,
)

print("Loading CLIP ViT-B/32 ...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("Loading Wav2Vec2 ...")
w2v_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE).eval()
w2v_fe = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")

print("Loading XLM-RoBERTa ...")
xlmr_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
xlmr_model = AutoModel.from_pretrained("xlm-roberta-base").to(DEVICE).eval()

print("Loading Qwen embedding model ...")
QWEN_NAME = "Qwen/Qwen3-Embedding-0.6B"
qwen_tok = AutoTokenizer.from_pretrained(QWEN_NAME)
qwen_model = AutoModel.from_pretrained(QWEN_NAME, dtype=torch.float16).to(DEVICE).eval()

for m in [clip_model, w2v_model, xlmr_model, qwen_model]:
    for p in m.parameters():
        p.requires_grad_(False)

# confirm real Qwen output dim
with torch.no_grad():
    _p = qwen_tok("test", return_tensors="pt", truncation=True, max_length=16, padding="max_length").to(DEVICE)
    QWEN_DIM = qwen_model(**_p).last_hidden_state.shape[-1]
print("QWEN_DIM =", QWEN_DIM)

CLIP_DIM, W2V_DIM, XLMR_DIM = 512, 768, 768
DIMS = {"vis": CLIP_DIM, "aud": W2V_DIM, "txl": XLMR_DIM, "tqw": QWEN_DIM}
print("Encoders ready. DIMS =", DIMS)

Loading CLIP ViT-B/32 ...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

QWEN_DIM = 1024
Encoders ready. DIMS = {'vis': 512, 'aud': 768, 'txl': 768, 'tqw': 1024}


# Cell 4 — Feature extraction functions

In [4]:
def clip_patch_features(pixel_values):
    vout = clip_model.vision_model(pixel_values=pixel_values)
    return vout.last_hidden_state, vout.pooler_output

@torch.no_grad()
def extract_visual(frame_dir):
    paths = sorted(glob.glob(os.path.join(frame_dir, "*.jpg")))
    imgs = [Image.open(p).convert("RGB") for p in paths]
    inp = clip_proc(images=imgs, return_tensors="pt").to(DEVICE)
    last_hidden, pooled = clip_patch_features(inp["pixel_values"])
    cls_token = last_hidden[:, 0, :]
    feats = clip_model.visual_projection(cls_token)
    feats = F.normalize(feats, dim=-1)
    return feats.cpu().float().numpy()   # (16, 512)

@torch.no_grad()
def extract_audio_from_wav(audio_path):
    import librosa
    try:
        wav, sr = librosa.load(audio_path, sr=16000, mono=True)
    except Exception:
        return np.zeros(W2V_DIM, dtype=np.float32)
    if wav is None or wav.size == 0:
        return np.zeros(W2V_DIM, dtype=np.float32)
    wav = wav[:16000*20]
    inp = w2v_fe(wav, sampling_rate=16000, return_tensors="pt").to(DEVICE)
    out = w2v_model(**inp).last_hidden_state
    return out.mean(dim=1).squeeze(0).cpu().float().numpy()

@torch.no_grad()
def extract_audio_from_video(video_path):
    import librosa
    try:
        wav, sr = librosa.load(video_path, sr=16000, mono=True)
    except Exception:
        return np.zeros(W2V_DIM, dtype=np.float32)
    if wav is None or wav.size == 0:
        return np.zeros(W2V_DIM, dtype=np.float32)
    wav = wav[:16000*20]
    inp = w2v_fe(wav, sampling_rate=16000, return_tensors="pt").to(DEVICE)
    out = w2v_model(**inp).last_hidden_state
    return out.mean(dim=1).squeeze(0).cpu().float().numpy()

def _mean_pool(h, mask):
    mask = mask.unsqueeze(-1).float()
    return (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def extract_text_xlmr(text):
    inp = xlmr_tok(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length").to(DEVICE)
    out = xlmr_model(**inp)
    return _mean_pool(out.last_hidden_state, inp["attention_mask"]).squeeze(0).cpu().float().numpy()

@torch.no_grad()
def extract_text_qwen(text):
    inp = qwen_tok(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length").to(DEVICE)
    out = qwen_model(**inp)
    return _mean_pool(out.last_hidden_state, inp["attention_mask"]).squeeze(0).cpu().float().numpy()

def safe_text(row):
    t = row.get("transcript", "")
    o = row.get("ocr_text", "")
    t = "" if (t is None or (isinstance(t, float) and pd.isna(t))) else str(t)
    o = "" if (o is None or (isinstance(o, float) and pd.isna(o))) else str(o)
    combined = (t + " " + o).strip().lower()
    return combined if combined else "no text available"

print("Extraction functions ready.")

Extraction functions ready.


# Cell 5 — Build Crossmodal dataframe + smoke test

In [5]:
CM_TXT = os.path.join(CM_ROOT, "dataset_kaggle", "extracted_text")
CM_FRAMES = os.path.join(CM_ROOT, "dataset_kaggle", "extracted_frames")
CM_AUDIO = os.path.join(CM_ROOT, "extracted_audio", "extracted_audio")

SUBMAP = {"identity_fabrication":0, "perception_manipulation":1,
          "scientifically_unrealistic_scene":2, "surreal_content":3}

def load_cm_split(split):
    df = pd.read_csv(os.path.join(CM_TXT, f"{split}_metadata.csv"))
    df["split"] = split
    df["label"] = (df["category"].str.strip().str.lower() == "misleading").astype(int)
    df["sub"] = df["subcategory"].map(SUBMAP).fillna(-1).astype(int)
    df["frame_dir"] = df.apply(lambda r: os.path.join(CM_FRAMES, split, r["video_id"]), axis=1)
    df["audio_path"] = df.apply(lambda r: os.path.join(CM_AUDIO, split, r["video_id"] + ".wav"), axis=1)
    return df

cm_train_df = load_cm_split("train")
cm_test_df = load_cm_split("test")
print("CM train:", len(cm_train_df), "| CM test:", len(cm_test_df))
print("label balance train:", cm_train_df["label"].value_counts().to_dict())

# smoke test
r = cm_train_df.iloc[0]
v = extract_visual(r["frame_dir"])
a = extract_audio_from_wav(r["audio_path"])
xh = extract_text_xlmr(safe_text(r))
qh = extract_text_qwen(safe_text(r))
print("smoke test -> vis:", v.shape, "aud:", a.shape, "xlmr:", xh.shape, "qwen:", qh.shape)

CM train: 1600 | CM test: 800
label balance train: {1: 800, 0: 800}
smoke test -> vis: (16, 512) aud: (768,) xlmr: (768,) qwen: (1024,)


# Cell 6 — Build FakeTT dataframe + smoke test

In [6]:
FT_VIDEO = os.path.join(FT_ROOT, "video")
FT_JSON = os.path.join(FT_ROOT, "data.json")

records = []
with open(FT_JSON) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

ft_df = pd.DataFrame(records)
ft_df["label"] = (ft_df["annotation"].str.strip().str.lower() == "fake").astype(int)
ft_df["video_path"] = ft_df["video_id"].apply(lambda x: os.path.join(FT_VIDEO, str(x) + ".mp4"))
ft_df = ft_df[ft_df["video_path"].apply(os.path.isfile)].reset_index(drop=True)

def ft_text(row):
    parts = []
    for k in ["description", "event"]:
        v = row.get(k, "")
        if isinstance(v, str) and v.strip():
            parts.append(v.strip())
    txt = " ".join(parts).lower()
    return txt if txt else "no text"
ft_df["text"] = ft_df.apply(ft_text, axis=1)

from sklearn.model_selection import train_test_split
tr_idx, te_idx = train_test_split(np.arange(len(ft_df)), test_size=0.2,
                                  stratify=ft_df["label"], random_state=SEED)
ft_df["split"] = "train"
ft_df.loc[te_idx, "split"] = "test"
print("FT total:", len(ft_df), "| train:", (ft_df.split=='train').sum(), "| test:", (ft_df.split=='test').sum())

# smoke test — frame sampling + audio from raw video
import cv2
def sample_frames_from_video(path, n=16):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return None
    idxs = set(np.linspace(0, max(total-1,0), n).astype(int).tolist())
    frames, i, grabbed = 0, 0, {}
    while True:
        ret, frame = cap.read()
        if not ret: break
        if i in idxs:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            grabbed[i] = Image.fromarray(frame)
        i += 1
        if len(grabbed) == len(idxs): break
    cap.release()
    if not grabbed: return None
    out = [grabbed[k] for k in sorted(grabbed.keys())]
    while len(out) < n: out.append(out[-1])
    return out[:n]

@torch.no_grad()
def clip_embed_frames(frames):
    inp = clip_proc(images=frames, return_tensors="pt").to(DEVICE)
    _, pooled = clip_patch_features(inp["pixel_values"])
    feats = clip_model.visual_projection(pooled)
    return F.normalize(feats, dim=-1).cpu().float().numpy()

r = ft_df.iloc[0]
fr = sample_frames_from_video(r["video_path"])
print("FT frames:", None if fr is None else len(fr))
if fr:
    print("FT clip emb:", clip_embed_frames(fr).shape)
print("FT audio emb:", extract_audio_from_video(r["video_path"]).shape)

FT total: 1992 | train: 1593 | test: 399
FT frames: 16
FT clip emb: (16, 512)


/tmp/ipykernel_24/113638639.py:34: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(video_path, sr=16000, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FT audio emb: (768,)


# Cell 7 — Full extraction: Crossmodal (resumable)

In [7]:
import gc

CM_WORK = os.path.join(WORK, "crossmodal")
os.makedirs(CM_WORK, exist_ok=True)

def extract_cm_split(df, split):
    N = len(df)
    ckpt_path = os.path.join(CM_WORK, f"cm_{split}_partial.npz")
    vis = np.zeros((N, 16, CLIP_DIM), dtype=np.float32)
    aud = np.zeros((N, W2V_DIM), dtype=np.float32)
    txl = np.zeros((N, XLMR_DIM), dtype=np.float32)
    tqw = np.zeros((N, QWEN_DIM), dtype=np.float32)
    done = np.zeros(N, dtype=bool)

    if os.path.exists(ckpt_path):
        ck = np.load(ckpt_path, allow_pickle=True)
        vis, aud, txl, tqw, done = ck["vis"], ck["aud"], ck["txl"], ck["tqw"], ck["done"]
        print(f"Resumed {split}: {done.sum()}/{N} already done")

    def save_ckpt():
        np.savez_compressed(ckpt_path, vis=vis, aud=aud, txl=txl, tqw=tqw, done=done)

    for i in tqdm(range(N), desc=f"extract CM {split}"):
        if done[i]:
            continue
        row = df.iloc[i]
        try:
            vis[i] = extract_visual(row["frame_dir"])
            aud[i] = extract_audio_from_wav(row["audio_path"])
            text = safe_text(row)
            txl[i] = extract_text_xlmr(text)
            tqw[i] = extract_text_qwen(text)
        except Exception as e:
            print(f"  failed idx {i} ({row['video_id']}): {e}")
        done[i] = True
        if (i + 1) % 200 == 0:
            save_ckpt(); torch.cuda.empty_cache(); gc.collect()

    save_ckpt()
    out_path = os.path.join(CM_WORK, f"cm_{split}_final.npz")
    np.savez_compressed(out_path, vis=vis, aud=aud, txl=txl, tqw=tqw,
                        lab=df["label"].values, sub=df["sub"].values,
                        vids=df["video_id"].values)
    print(f"Saved {out_path} | shapes: vis{vis.shape} aud{aud.shape} txl{txl.shape} tqw{tqw.shape}")
    return out_path

extract_cm_split(cm_train_df, "train")
extract_cm_split(cm_test_df, "test")
print("\nCROSSMODAL EXTRACTION COMPLETE.")

extract CM train:   0%|          | 0/1600 [00:00<?, ?it/s]

Saved /kaggle/working/crossmodal/cm_train_final.npz | shapes: vis(1600, 16, 512) aud(1600, 768) txl(1600, 768) tqw(1600, 1024)


extract CM test:   0%|          | 0/800 [00:00<?, ?it/s]

Saved /kaggle/working/crossmodal/cm_test_final.npz | shapes: vis(800, 16, 512) aud(800, 768) txl(800, 768) tqw(800, 1024)

CROSSMODAL EXTRACTION COMPLETE.


# Cell 8 — Full extraction: FakeTT (resumable)

In [8]:
FT_WORK = os.path.join(WORK, "fakett")
os.makedirs(FT_WORK, exist_ok=True)

def extract_ft_all(df):
    N = len(df)
    ckpt_path = os.path.join(FT_WORK, "ft_partial.npz")
    vis = np.zeros((N, 16, CLIP_DIM), dtype=np.float32)
    aud = np.zeros((N, W2V_DIM), dtype=np.float32)
    txl = np.zeros((N, XLMR_DIM), dtype=np.float32)
    tqw = np.zeros((N, QWEN_DIM), dtype=np.float32)
    done = np.zeros(N, dtype=bool)
    fail_ids = []

    if os.path.exists(ckpt_path):
        ck = np.load(ckpt_path, allow_pickle=True)
        vis, aud, txl, tqw, done = ck["vis"], ck["aud"], ck["txl"], ck["tqw"], ck["done"]
        print(f"Resumed FT: {done.sum()}/{N} already done")

    def save_ckpt():
        np.savez_compressed(ckpt_path, vis=vis, aud=aud, txl=txl, tqw=tqw, done=done)

    for i in tqdm(range(N), desc="extract FakeTT"):
        if done[i]:
            continue
        row = df.iloc[i]
        try:
            frames = sample_frames_from_video(row["video_path"], n=16)
            if frames is None:
                fail_ids.append(row["video_id"])
            else:
                vis[i] = clip_embed_frames(frames)
            aud[i] = extract_audio_from_video(row["video_path"])
            txl[i] = extract_text_xlmr(row["text"])
            tqw[i] = extract_text_qwen(row["text"])
        except Exception as e:
            fail_ids.append(row["video_id"])
        done[i] = True
        if (i + 1) % 200 == 0:
            save_ckpt(); torch.cuda.empty_cache(); gc.collect()

    save_ckpt()
    print(f"\nExtraction complete. Failures (zero-filled): {len(fail_ids)}")

    tr_mask = (df["split"] == "train").values
    te_mask = (df["split"] == "test").values
    np.savez_compressed(os.path.join(FT_WORK, "ft_train_final.npz"),
        vis=vis[tr_mask], aud=aud[tr_mask], txl=txl[tr_mask], tqw=tqw[tr_mask], lab=df["label"].values[tr_mask])
    np.savez_compressed(os.path.join(FT_WORK, "ft_test_final.npz"),
        vis=vis[te_mask], aud=aud[te_mask], txl=txl[te_mask], tqw=tqw[te_mask], lab=df["label"].values[te_mask])
    print("Saved ft_train_final.npz:", tr_mask.sum(), "| ft_test_final.npz:", te_mask.sum())

extract_ft_all(ft_df)
print("\nFAKETT EXTRACTION COMPLETE.")

extract FakeTT:   0%|          | 0/1992 [00:00<?, ?it/s]

/tmp/ipykernel_24/113638639.py:34: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(video_path, sr=16000, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_24/113638639.py:34: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(video_path, sr=16000, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_24/113638639.py:34: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(video_path, sr=16000, mono=True)
/usr/l


Extraction complete. Failures (zero-filled): 0
Saved ft_train_final.npz: 1593 | ft_test_final.npz: 399

FAKETT EXTRACTION COMPLETE.


In [9]:
def load_npz_dict(path, has_sub=False):
    d = np.load(path, allow_pickle=True)
    out = {
        "vis_seq": torch.tensor(d["vis"]),
        "vis": torch.tensor(d["vis"].mean(1)),
        "aud": torch.tensor(d["aud"]),
        "txl": torch.tensor(d["txl"]),
        "tqw": torch.tensor(d["tqw"]),
        "lab": torch.tensor(d["lab"]).long(),
    }
    if has_sub:
        out["sub"] = torch.tensor(d["sub"]).long()
    return out

CM_TR = load_npz_dict(os.path.join(CM_WORK, "cm_train_final.npz"), has_sub=True)
CM_TE = load_npz_dict(os.path.join(CM_WORK, "cm_test_final.npz"), has_sub=True)
FT_TR = load_npz_dict(os.path.join(FT_WORK, "ft_train_final.npz"))
FT_TE = load_npz_dict(os.path.join(FT_WORK, "ft_test_final.npz"))

print("CM train:", CM_TR["lab"].shape[0], "| CM test:", CM_TE["lab"].shape[0])
print("FT train:", FT_TR["lab"].shape[0], "| FT test:", FT_TE["lab"].shape[0])

def misleading_mask(batch): return batch["sub"] >= 0
SUBNAMES = ["IF", "PM", "SUS", "SC"]

CM train: 1600 | CM test: 800
FT train: 1593 | FT test: 399


In [10]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef)

class ModalityProj(nn.Module):
    def __init__(self, in_dim, out_dim=384, p=0.4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(p))
    def forward(self, x): return self.net(x)

class TemporalAttn(nn.Module):
    def __init__(self, dim=512):
        super().__init__(); self.w = nn.Linear(dim, 1)
    def forward(self, seq):
        a = torch.softmax(self.w(seq).squeeze(-1), dim=1)
        return (a.unsqueeze(-1) * seq).sum(1)

class CrossAttn(nn.Module):
    def __init__(self, dim=384, heads=4):
        super().__init__(); self.mha = nn.MultiheadAttention(dim, heads, batch_first=True)
    def forward(self, q, kv):
        q, kv = q.unsqueeze(1), kv.unsqueeze(1)
        o, _ = self.mha(q, kv, kv)
        return o.squeeze(1)

class MultiKeyCrossAttn(nn.Module):
    def __init__(self, dim=384, heads=4):
        super().__init__(); self.mha = nn.MultiheadAttention(dim, heads, batch_first=True)
    def forward(self, q, kv_stack):
        q = q.unsqueeze(1)
        o, _ = self.mha(q, kv_stack, kv_stack)
        return o.squeeze(1)

def move(batch, idx=None):
    out = {}
    for k, v in batch.items():
        out[k] = (v[idx] if idx is not None else v).to(DEVICE)
    return out

def compute_metrics(y_true, y_pred, y_prob=None, n_classes=2):
    m = {
        "Accuracy": accuracy_score(y_true, y_pred)*100,
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)*100,
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    if n_classes == 2 and y_prob is not None:
        m["ROC-AUC"] = roc_auc_score(y_true, y_prob)
    elif y_prob is not None:
        try: m["ROC-AUC"] = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
        except Exception: m["ROC-AUC"] = float("nan")
    return m

class FusionModelV2(nn.Module):
    def __init__(self, mods=["vis","aud","txl","tqw"], mode="cross", d=384, dropout=0.4,
                 n_classes=2, use_temporal=True):
        super().__init__()
        self.mods, self.mode, self.d = mods, mode, d
        self.use_temporal = use_temporal
        M = len(mods)
        if "vis" in mods and use_temporal:
            self.tattn = TemporalAttn(512)
        self.proj = nn.ModuleDict({m: ModalityProj(DIMS[m], d, dropout) for m in mods})

        if mode in ("cross", "proposed") and M >= 2:
            self.cross = CrossAttn(d)
        if mode in ("gated", "proposed"):
            self.gate = nn.Sequential(nn.Linear(d*M, d*M), nn.Sigmoid())
        if mode in ("attn", "proposed"):
            self.wgt = nn.Linear(d*M, M)
        if mode == "quafusion":
            self.quality_heads = nn.ModuleDict({m: nn.Sequential(nn.Linear(d,32), nn.ReLU(), nn.Linear(32,1)) for m in mods})
            self.refine_cross = MultiKeyCrossAttn(d)
        if mode == "ccinsa":
            self.cross_per_mod = nn.ModuleDict({m: CrossAttn(d) for m in mods})
            self.agg_wgt = nn.Linear(d*M, M)
        if mode == "ambiguity":
            n_pairs = M*(M-1)//2
            self.amb_mlp = nn.Sequential(nn.Linear(n_pairs,32), nn.ReLU(), nn.Linear(32,M))
            if M >= 2: self.base_cross = CrossAttn(d)
        if mode == "atcaf":
            if M >= 2:
                self.cross_obs = CrossAttn(d)
                self.cross_spurious = CrossAttn(d)
            self.lambda_causal = nn.Parameter(torch.tensor(0.0))

        self.head = nn.Sequential(nn.Linear(d,128), nn.LayerNorm(128), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_classes))

    def encode(self, batch):
        feats = {}
        for m in self.mods:
            feats[m] = self.proj[m](self.tattn(batch["vis_seq"])) if (m=="vis" and self.use_temporal) else self.proj[m](batch[m])
        return feats

    def forward(self, batch):
        f = self.encode(batch); mats = [f[m] for m in self.mods]; M = len(mats)
        cat = torch.cat(mats, dim=-1)

        if self.mode == "concat": fused = cat
        elif self.mode == "gated": fused = self.gate(cat) * cat
        elif self.mode == "attn":
            w = torch.softmax(self.wgt(cat), dim=-1)
            fused = sum(w[:,i:i+1]*mats[i] for i in range(M))
        elif self.mode == "cross":
            others = sum(mats[1:])/max(M-1,1) if M>=2 else mats[0]
            fused = (self.cross(mats[0], others)+mats[0]) if M>=2 else mats[0]
        elif self.mode == "proposed":
            others = sum(mats[1:])/max(M-1,1) if M>=2 else mats[0]
            fcross = (self.cross(mats[0], others)+mats[0]) if M>=2 else mats[0]
            fgated = (self.gate(cat)*cat).view(cat.size(0),M,self.d).mean(1)
            w = torch.softmax(self.wgt(cat), dim=-1)
            fadapt = sum(w[:,i:i+1]*mats[i] for i in range(M))
            fused = fcross+fgated+fadapt
        elif self.mode == "quafusion":
            q = torch.cat([torch.sigmoid(self.quality_heads[m](f[m])) for m in self.mods], dim=1)
            q = torch.softmax(q, dim=1)
            f_weighted = sum(q[:,i:i+1]*mats[i] for i in range(M))
            stacked = torch.stack(mats, dim=1)
            fused = self.refine_cross(f_weighted, stacked) + f_weighted
        elif self.mode == "ccinsa":
            c_list = []
            for i,m in enumerate(self.mods):
                other_mean = sum(mats[j] for j in range(M) if j!=i)/max(M-1,1)
                c_list.append(self.cross_per_mod[m](mats[i], other_mean)+mats[i])
            cat_c = torch.cat(c_list, dim=-1)
            w = torch.softmax(self.agg_wgt(cat_c), dim=-1)
            fused = sum(w[:,i:i+1]*c_list[i] for i in range(M))
        elif self.mode == "ambiguity":
            pairs = []
            for i in range(M):
                for j in range(i+1,M):
                    pairs.append((1-F.cosine_similarity(mats[i],mats[j],dim=-1,eps=1e-8)).unsqueeze(1))
            amb_stack = torch.cat(pairs, dim=1)
            adj = torch.softmax(self.amb_mlp(amb_stack), dim=1)
            f_weighted = sum(adj[:,i:i+1]*mats[i] for i in range(M))
            others = sum(mats[1:])/max(M-1,1) if M>=2 else mats[0]
            base = (self.base_cross(mats[0], others)+mats[0]) if M>=2 else mats[0]
            fused = base+f_weighted
        elif self.mode == "atcaf":
            others = sum(mats[1:])/max(M-1,1) if M>=2 else mats[0]
            if M>=2:
                F_obs = self.cross_obs(mats[0], others)+mats[0]
                perm = torch.randperm(others.size(0), device=others.device)
                F_spurious = self.cross_spurious(mats[0], others[perm])
                fused = F_obs - torch.sigmoid(self.lambda_causal)*F_spurious
            else: fused = mats[0]
        return self.head(fused)

def train_eval_v2(mods, train, evalset, mode="cross", n_classes=2, label_key="lab",
                  d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60, bs=64,
                  use_temporal=True, seed=42, filter_fn=None, return_model=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FusionModelV2(mods, mode=mode, d=d, dropout=dropout, n_classes=n_classes, use_temporal=use_temporal).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss()
    tr = train
    if filter_fn is not None:
        mask = filter_fn(tr); tr = {k:v[mask] for k,v in tr.items()}
    N = tr[label_key].shape[0]; idx_all = np.arange(N)
    for ep in range(epochs):
        model.train(); np.random.shuffle(idx_all)
        for s in range(0,N,bs):
            b = move(tr, idx_all[s:s+bs])
            loss = lossf(model(b), b[label_key])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    ev = evalset
    if filter_fn is not None:
        mask = filter_fn(ev); ev = {k:v[mask] for k,v in ev.items()}
    model.eval()
    with torch.no_grad():
        b = move(ev); logits = model(b)
        prob = torch.softmax(logits,1).cpu().numpy()
        pred = logits.argmax(1).cpu().numpy(); true = ev[label_key].numpy()
    prob_auc = prob[:,1] if n_classes==2 else prob
    m = compute_metrics(true, pred, prob_auc, n_classes)
    return (m, model) if return_model else (m, None)

print("Cell 10 ready.")

Cell 10 ready.


In [11]:
import itertools, csv

log_rows = []  # collect everything for the final CSV/zip

def carve_val(train_dict, label_key, frac=0.15, seed=42):
    N = train_dict[label_key].shape[0]
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(N, generator=g)
    n_val = int(frac*N)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    return {k:v[tr_idx] for k,v in train_dict.items()}, {k:v[val_idx] for k,v in train_dict.items()}

CM_TRsub, CM_VAL = carve_val(CM_TR, "lab")
FT_TRsub, FT_VAL = carve_val(FT_TR, "lab")

grid = {"lr":[1e-3, 2e-3], "dropout":[0.2, 0.4]}
methods = ["quafusion", "ccinsa", "ambiguity", "atcaf"]
best_hp = {}

def search_method(mode, train, val, n_classes, label_key, filter_fn=None):
    best = None
    for lr, dp in itertools.product(grid["lr"], grid["dropout"]):
        m, _ = train_eval_v2(["vis","aud","txl","tqw"], train, val, mode=mode,
                             n_classes=n_classes, label_key=label_key, d=384, dropout=dp,
                             lr=lr, wd=1e-2, epochs=60, filter_fn=filter_fn, seed=42)
        row = {"mode":mode, "lr":lr, "dropout":dp, "val_acc":m["Accuracy"], "val_f1":m["Macro F1"]}
        log_rows.append(row)
        if best is None or m["Accuracy"] > best[0]:
            best = (m["Accuracy"], lr, dp)
    return best

print("=== Hyperparameter search: Stage 1 ===")
for mode in methods:
    acc, lr, dp = search_method(mode, CM_TRsub, CM_VAL, 2, "lab")
    best_hp[("s1", mode)] = {"lr":lr, "dropout":dp}
    print(f"  {mode:12s} best val_acc={acc:.2f}  lr={lr}  dropout={dp}")

print("\n=== Hyperparameter search: Stage 2 ===")
CM_TRsub_mis = {k:v[misleading_mask(CM_TRsub)] for k,v in CM_TRsub.items()}
CM_VAL_mis = {k:v[misleading_mask(CM_VAL)] for k,v in CM_VAL.items()}
for mode in methods:
    acc, lr, dp = search_method(mode, CM_TRsub_mis, CM_VAL_mis, 4, "sub")
    best_hp[("s2", mode)] = {"lr":lr, "dropout":dp}
    print(f"  {mode:12s} best val_acc={acc:.2f}  lr={lr}  dropout={dp}")

print("\n=== Hyperparameter search: FakeTT ===")
for mode in methods:
    acc, lr, dp = search_method(mode, FT_TRsub, FT_VAL, 2, "lab")
    best_hp[("ft", mode)] = {"lr":lr, "dropout":dp}
    print(f"  {mode:12s} best val_acc={acc:.2f}  lr={lr}  dropout={dp}")

print("\nSearch complete. best_hp =", best_hp)

=== Hyperparameter search: Stage 1 ===
  quafusion    best val_acc=97.50  lr=0.001  dropout=0.4
  ccinsa       best val_acc=98.33  lr=0.001  dropout=0.4
  ambiguity    best val_acc=97.92  lr=0.002  dropout=0.4
  atcaf        best val_acc=97.92  lr=0.001  dropout=0.4

=== Hyperparameter search: Stage 2 ===
  quafusion    best val_acc=91.87  lr=0.001  dropout=0.2
  ccinsa       best val_acc=93.50  lr=0.001  dropout=0.4
  ambiguity    best val_acc=94.31  lr=0.001  dropout=0.4
  atcaf        best val_acc=93.50  lr=0.001  dropout=0.2

=== Hyperparameter search: FakeTT ===
  quafusion    best val_acc=81.93  lr=0.001  dropout=0.4
  ccinsa       best val_acc=83.19  lr=0.002  dropout=0.4
  ambiguity    best val_acc=85.29  lr=0.001  dropout=0.2
  atcaf        best val_acc=84.03  lr=0.001  dropout=0.2

Search complete. best_hp = {('s1', 'quafusion'): {'lr': 0.001, 'dropout': 0.4}, ('s1', 'ccinsa'): {'lr': 0.001, 'dropout': 0.4}, ('s1', 'ambiguity'): {'lr': 0.002, 'dropout': 0.4}, ('s1', 'atcaf'):

In [12]:
def multi_seed_eval(mode, train, test, n_classes, label_key, hp, filter_fn=None, seeds=(42,43,44)):
    accs, f1s, mccs = [], [], []
    for s in seeds:
        m, _ = train_eval_v2(["vis","aud","txl","tqw"], train, test, mode=mode,
                             n_classes=n_classes, label_key=label_key, d=384,
                             dropout=hp["dropout"], lr=hp["lr"], wd=1e-2, epochs=60,
                             filter_fn=filter_fn, seed=s)
        accs.append(m["Accuracy"]); f1s.append(m["Macro F1"]); mccs.append(m["MCC"])
    return np.mean(accs), np.std(accs), np.mean(f1s), np.mean(mccs)

final_results = []
tasks = [
    ("Stage 1", CM_TR, CM_TE, 2, "lab", None, "s1"),
    ("Stage 2", CM_TR, CM_TE, 4, "sub", misleading_mask, "s2"),
    ("FakeTT",  FT_TR, FT_TE, 2, "lab", None, "ft"),
]

print("=== FINAL: tuned, 3-seed comparison, all methods, all tasks ===\n")
for task_name, tr, te, ncls, lkey, ffn, key in tasks:
    print(f"--- {task_name} ---")
    base_hp = {"lr":2e-3, "dropout":0.4}
    acc_m, acc_s, f1_m, mcc_m = multi_seed_eval("cross", tr, te, ncls, lkey, base_hp, ffn)
    print(f"  {'Cross-Attention':16s} Acc={acc_m:.2f}±{acc_s:.2f}  MacroF1={f1_m:.2f}  MCC={mcc_m:.3f}")
    final_results.append([task_name,"Cross-Attention",acc_m,acc_s,f1_m,mcc_m])
    for mode in methods:
        hp = best_hp[(key, mode)]
        acc_m, acc_s, f1_m, mcc_m = multi_seed_eval(mode, tr, te, ncls, lkey, hp, ffn)
        print(f"  {mode:16s} Acc={acc_m:.2f}±{acc_s:.2f}  MacroF1={f1_m:.2f}  MCC={mcc_m:.3f}  (lr={hp['lr']}, dropout={hp['dropout']})")
        final_results.append([task_name,mode,acc_m,acc_s,f1_m,mcc_m])
    print()

# save results
with open(os.path.join(WORK, "final_fusion_comparison.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Task","Method","Mean_Acc","Std_Acc","Mean_MacroF1","Mean_MCC"])
    w.writerows(final_results)

with open(os.path.join(WORK, "hyperparam_search_log.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["mode","lr","dropout","val_acc","val_f1"])
    w.writeheader(); w.writerows(log_rows)

print("Saved final_fusion_comparison.csv and hyperparam_search_log.csv")

=== FINAL: tuned, 3-seed comparison, all methods, all tasks ===

--- Stage 1 ---
  Cross-Attention  Acc=87.71±0.56  MacroF1=87.71  MCC=0.754
  quafusion        Acc=87.38±0.71  MacroF1=87.37  MCC=0.748  (lr=0.001, dropout=0.4)
  ccinsa           Acc=87.88±0.00  MacroF1=87.87  MCC=0.758  (lr=0.001, dropout=0.4)
  ambiguity        Acc=86.50±0.57  MacroF1=86.50  MCC=0.730  (lr=0.002, dropout=0.4)
  atcaf            Acc=87.62±0.54  MacroF1=87.62  MCC=0.753  (lr=0.001, dropout=0.4)

--- Stage 2 ---
  Cross-Attention  Acc=82.25±0.41  MacroF1=82.30  MCC=0.767
  quafusion        Acc=80.42±1.05  MacroF1=80.55  MCC=0.745  (lr=0.001, dropout=0.2)
  ccinsa           Acc=80.58±2.60  MacroF1=80.45  MCC=0.747  (lr=0.001, dropout=0.4)
  ambiguity        Acc=84.17±0.66  MacroF1=84.29  MCC=0.792  (lr=0.001, dropout=0.4)
  atcaf            Acc=83.25±1.22  MacroF1=83.42  MCC=0.780  (lr=0.001, dropout=0.2)

--- FakeTT ---
  Cross-Attention  Acc=83.63±0.31  MacroF1=82.89  MCC=0.660
  quafusion        Acc=84.

In [13]:
import shutil, json as _json

# save the overall best model per task (whichever architecture wins)
os.makedirs(os.path.join(WORK, "checkpoints"), exist_ok=True)
best_per_task = {}
for task_name, tr, te, ncls, lkey, ffn, key in tasks:
    rows = [r for r in final_results if r[0] == task_name]
    best_row = max(rows, key=lambda r: r[2])  # highest mean acc
    best_method = best_row[1]
    hp = {"lr":2e-3,"dropout":0.4} if best_method=="Cross-Attention" else best_hp[(key, best_method)]
    _, model = train_eval_v2(["vis","aud","txl","tqw"], tr, te, mode=("cross" if best_method=="Cross-Attention" else best_method),
                             n_classes=ncls, label_key=lkey, d=384, dropout=hp["dropout"], lr=hp["lr"],
                             wd=1e-2, epochs=60, filter_fn=ffn, seed=42, return_model=True)
    ckpt_path = os.path.join(WORK, "checkpoints", f"best_{key}_{best_method}.pth")
    torch.save(model.state_dict(), ckpt_path)
    best_per_task[task_name] = {"method": best_method, "hp": hp, "accuracy": best_row[2]}
    print(f"{task_name}: best={best_method} (Acc={best_row[2]:.2f}) -> saved {ckpt_path}")

with open(os.path.join(WORK, "best_methods_summary.json"), "w") as f:
    _json.dump(best_per_task, f, indent=2)

# bundle everything into one zip
BUNDLE = os.path.join(WORK, "fusion_experiment_bundle")
os.makedirs(BUNDLE, exist_ok=True)
for fname in ["final_fusion_comparison.csv", "hyperparam_search_log.csv", "best_methods_summary.json"]:
    shutil.copy(os.path.join(WORK, fname), BUNDLE)
shutil.copytree(os.path.join(WORK, "checkpoints"), os.path.join(BUNDLE, "checkpoints"), dirs_exist_ok=True)
# include the extracted features too, so nothing needs re-extracting later
shutil.copytree(CM_WORK, os.path.join(BUNDLE, "crossmodal_features"), dirs_exist_ok=True)
shutil.copytree(FT_WORK, os.path.join(BUNDLE, "fakett_features"), dirs_exist_ok=True)

shutil.make_archive("/kaggle/working/fusion_experiment_results", "zip", BUNDLE)
print("\nDONE. Download /kaggle/working/fusion_experiment_results.zip from the Output tab after commit finishes.")
print("Contents: final_fusion_comparison.csv, hyperparam_search_log.csv, best_methods_summary.json, checkpoints/, crossmodal_features/, fakett_features/")

Stage 1: best=ccinsa (Acc=87.88) -> saved /kaggle/working/checkpoints/best_s1_ccinsa.pth
Stage 2: best=ambiguity (Acc=84.17) -> saved /kaggle/working/checkpoints/best_s2_ambiguity.pth
FakeTT: best=ambiguity (Acc=84.80) -> saved /kaggle/working/checkpoints/best_ft_ambiguity.pth

DONE. Download /kaggle/working/fusion_experiment_results.zip from the Output tab after commit finishes.
Contents: final_fusion_comparison.csv, hyperparam_search_log.csv, best_methods_summary.json, checkpoints/, crossmodal_features/, fakett_features/
